In [58]:
import os 

In [59]:
%pwd

'd:\\Cdac_project\\Wine_prediction_e2e'

In [68]:
os.chdir(r"D:\Cdac_project\Wine_prediction_e2e")

In [69]:
%pwd

'D:\\Cdac_project\\Wine_prediction_e2e'

In [74]:
#creating entity here custom output

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict

In [75]:
# update the configuration manager in src 

from src.Wine_prediction_e2e.constants import *
from src.Wine_prediction_e2e.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = Config_yaml_path,
        params_filepath = params_yaml_path,
        schema_filepath = schema_yaml_path):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir = config.unzip_data_dir,
            all_schema=schema,
        )

        return data_validation_config

In [76]:
#update the component

import os
from Wine_prediction_e2e import logger
import pandas as pd

class DataValiadtion:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    
    def validate_all_columns(self)-> bool:
        try:
            validation_status = None
            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)

            all_schema = self.config.all_schema.keys()

            
            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")

            return validation_status
        
        except Exception as e:
            raise e

In [77]:
# Updae the pipeline

try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValiadtion(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

[2025-10-31 22:31:57,272: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-10-31 22:31:57,274: INFO: common: yaml file: params.yaml loaded successfully]
[2025-10-31 22:31:57,277: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-10-31 22:31:57,277: INFO: common: created directory at: artifacts]
[2025-10-31 22:31:57,278: INFO: common: created directory at: artifacts/data_validation]
